In [ ]:
import json

import pandas as pd

# https://huggingface.co/datasets/open-r1/OpenThoughts-114k-math

df = pd.read_parquet("open_thoughts_math_cot.parquet")
df

In [ ]:
short_df = df[(df['generated_token_count'] < 1024) & (df['correct'])]
short_df

In [ ]:
short_df['conversations']

In [ ]:
replacements = {
    "<|begin_of_thought|>": "<think>",
    "<|end_of_thought|>": "</think>",
    "<|begin_of_solution|>": "<answer>",
    "<|end_of_solution|>": "</answer>",
}

def multi_replace(text, mapping):
    for old, new in mapping.items():
        text = text.replace(old, new)
    return text

In [ ]:
short_df["response"] = short_df["conversations"].apply(
    lambda arr: next((item["value"] for item in arr if item["from"] == "assistant"), None)
).apply(lambda x: multi_replace(x, replacements)).apply(lambda x: x.replace("</think>\n\n<answer>", "</think> <answer>"))

short_df

In [ ]:
short_df.rename(columns={'problem': 'prompt'}, inplace=True)

In [ ]:
short_df = short_df[['prompt', 'response']]
short_df

In [ ]:
short_df.to_parquet('short_reasoning_86.parquet', index=False)